<a href="https://colab.research.google.com/github/nigevolution/bassspecmatchpro-colab/blob/main/BassSpecMatchPRO-NAM-Batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BassSpecMatchPRO NAM — fila persistente no Google Drive
O standalone grava os jobs no Drive. Reexecute a célula após reconexão; modelos já concluídos são reutilizados pelo hash do job.


In [ ]:
from google.colab import files, drive
import pathlib, zipfile, json, wave, subprocess, shutil, time, hashlib

def safe_extract(job, root):
    with zipfile.ZipFile(job) as z:
        for member in z.infolist():
            target = (root / member.filename).resolve()
            assert target == root.resolve() or root.resolve() in target.parents, "ZIP contém caminho inválido"
        z.extractall(root)

def wav_info(path):
    with wave.open(str(path), "rb") as w:
        info = (w.getframerate(), w.getnchannels(), w.getnframes(), w.getsampwidth())
        frames = w.readframes(w.getnframes())
    return info, frames

def validate_job(root):
    manifest_path = root / "manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("manifest.json ausente")
    manifest = json.loads(manifest_path.read_text())
    if manifest.get("format") != "bassspec-nam-colab-job-v7" or manifest.get("schema_version") != 7:
        raise ValueError("job/schema Colab v7 inválido")
    if manifest.get("training_mode") != "input-output" or manifest.get("transport") != "colab-direct-upload":
        raise ValueError("contrato de treino/upload inválido")
    if manifest.get("reference_timbre_only") is not True or manifest.get("contains_source_audio") is not False:
        raise ValueError("proveniência do job inválida")
    if (manifest.get("reference_conditioned_target") is not True
        or manifest.get("canonical_output_authority") is not True
        or manifest.get("conditioned_transfer") is not True
        or manifest.get("linear_exports_from_same_target") is not True
        or manifest.get("target_model") != "reference-conditioned-fidelity-v2"
        or manifest.get("stimulus_role") != "carrier-only"
        or manifest.get("fidelity_fir_method") != "h1-wiener-8192-v1"):
        raise ValueError("job sem target condicionado compartilhado de fidelidade")
    if manifest.get("contains_reference_audio") is not False or (root / "reference.wav").exists():
        raise ValueError("reference.wav bruto não deve estar no job")
    for name in ("input.wav", "output.wav", "model.nam", "train_nam_official.py"):
        if not (root / name).is_file():
            raise FileNotFoundError(name + " ausente")
    input_info, input_bytes = wav_info(root / "input.wav")
    output_info, output_bytes = wav_info(root / "output.wav")
    if input_info != output_info:
        raise ValueError("input/output incompatíveis")
    if input_info[0] != 48000 or input_info[1] != 1 or input_info[2] <= 0:
        raise ValueError("input/output devem ser 48 kHz mono e não vazios")
    if not any(input_bytes) or not any(output_bytes) or input_bytes == output_bytes:
        raise ValueError("input/output silencioso ou sem transformação")
    return manifest, input_info

def valid_model(path):
    try:
        data = json.loads(path.read_text())
        return data.get("architecture") == "SlimmableContainer" and data.get("version") == "0.7.0"
    except Exception:
        return False

print("BassSpecMatchPRO NAM — fila persistente no Google Drive")
drive.mount("/content/drive")
base = pathlib.Path("/content/drive/MyDrive/BassSpecMatchPRO NAM")
pointer = base / "current_queue.txt"
drive_mode = pointer.is_file()
if drive_mode:
    queue_id = pointer.read_text().strip()
    if not queue_id or pathlib.Path(queue_id).name != queue_id:
        raise ValueError("current_queue.txt inválido")
    queue_root = base / "Queues" / queue_id
    jobs_dir = queue_root / "Jobs"
    results_dir = queue_root / "Results"
    results_dir.mkdir(parents=True, exist_ok=True)
    zip_paths = sorted(jobs_dir.glob("*.zip"))
    if not zip_paths:
        raise FileNotFoundError(f"Nenhum ZIP em {jobs_dir}")
    print("Fila Drive:", queue_root)
else:
    print("Fila persistente não encontrada; usando upload manual como fallback.")
    uploaded = files.upload()
    zip_paths = [pathlib.Path(name) for name in uploaded if name.lower().endswith(".zip")]
    results_dir = pathlib.Path("/content")
assert zip_paths, "Nenhum job ZIP encontrado"

subprocess.run(["pip", "install", "-q", "neural-amp-modeler==0.13.0"], check=True)
import torch
assert torch.cuda.is_available(), "Ative GPU T4/CUDA em Ambiente de execução > Alterar tipo de ambiente"
print("GPU:", torch.cuda.get_device_name(0))
print(f"Fila recebida: {len(zip_paths)} job(s). Cada referência será treinada isoladamente.")

results = []
for job_index, job in enumerate(zip_paths, 1):
    job = pathlib.Path(job)
    job_sha = hashlib.sha256(job.read_bytes()).hexdigest()
    root = pathlib.Path(f"/content/bassspec_nam_job_{job_index}")
    shutil.rmtree(root, ignore_errors=True); root.mkdir()
    safe_extract(job, root)
    manifest, input_info = validate_job(root)
    epochs = int(manifest.get("epochs", manifest.get("default_epochs", 100)))
    assert 1 <= epochs <= 1000, "epochs fora do intervalo suportado (1..1000)"
    reference_name = str(manifest.get("reference_name") or job.stem).strip()
    reference_name = pathlib.Path(reference_name).name.strip() or f"Referencia_{job_index}"
    result_name = results_dir / (reference_name + ".nam")
    receipt = results_dir / (reference_name + ".job.sha256")
    if result_name.is_file() and receipt.is_file() and receipt.read_text().strip() == job_sha and valid_model(result_name):
        results.append(result_name)
        print(f"[{job_index}/{len(zip_paths)}] já concluído, reutilizando: {result_name.name}")
        continue

    print(f"[{job_index}/{len(zip_paths)}] {reference_name}: 48 kHz mono, {input_info[2]/48000:.1f}s, {epochs} epochs")
    preflight = pathlib.Path(f"/content/nam_preflight_{job_index}")
    shutil.rmtree(preflight, ignore_errors=True)
    cmd_preflight = ["python", str(root / "train_nam_official.py"), "--input", str(root / "input.wav"), "--output", str(root / "output.wav"), "--outdir", str(preflight), "--template", str(root / "model.nam"), "--preflight-only"]
    subprocess.run(cmd_preflight, check=True)

    out = pathlib.Path(f"/content/nam_result_{job_index}")
    shutil.rmtree(out, ignore_errors=True); out.mkdir()
    log_path = out / "trainer.log"
    cmd = ["python", str(root / "train_nam_official.py"), "--input", str(root / "input.wav"), "--output", str(root / "output.wav"), "--outdir", str(out), "--template", str(root / "model.nam"), "--epochs", str(epochs), "--ignore-checks"]
    print("TREINO REAL:", " ".join(cmd))
    with log_path.open("w") as log_file:
        proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, text=True)
        progress_path = out / "training-progress.json"
        last = None
        while proc.poll() is None:
            if progress_path.exists():
                try:
                    d = json.loads(progress_path.read_text())
                    stamp = (d.get("phase"), d.get("epoch"), d.get("epochs"), d.get("elapsed_seconds"))
                    if stamp != last:
                        print(f"[{job_index}/{len(zip_paths)}] {d.get('phase')} | epoch {d.get('epoch')}/{d.get('epochs')} | {d.get('elapsed_seconds')}s")
                        last = stamp
                except Exception:
                    pass
            time.sleep(2)
    if proc.returncode != 0:
        raise RuntimeError(f"Falha no job {job.name}:\n" + log_path.read_text(errors="replace")[-5000:])

    model = out / "trained.nam"
    if not valid_model(model):
        raise RuntimeError("trained.nam inválido")
    shutil.copy2(model, result_name)
    receipt.write_text(job_sha)
    if drive_mode:
        shutil.copy2(log_path, results_dir / (reference_name + ".trainer.log"))
        (queue_root / "queue-progress.json").write_text(json.dumps({
            "queue": queue_id, "completed": job_index, "total": len(zip_paths),
            "last_reference": reference_name, "updated_at": time.time()
        }, indent=2))
    results.append(result_name)
    print(f"[{job_index}/{len(zip_paths)}] concluído: {result_name.name}")

if drive_mode:
    print("Fila concluída. Modelos persistidos em:", results_dir)
    print("Se o navegador desconectar, execute esta célula novamente; modelos com o mesmo hash serão ignorados.")
else:
    print("Fila concluída. Baixando modelos individuais...")
    for model in results:
        files.download(str(model))


BassSpecMatchPRO NAM — fila persistente no Google Drive
Mounted at /content/drive
Fila Drive: /content/drive/MyDrive/BassSpecMatchPRO NAM/Queues/21-referencias-20260828-102318
GPU: Tesla T4
Fila recebida: 21 job(s). Cada referência será treinada isoladamente.
[1/21] já concluído, reutilizando: BASS MODS.nam
[2/21] já concluído, reutilizando: DMARK.nam
[3/21] já concluído, reutilizando: FACTOR.nam
[4/21] já concluído, reutilizando: FENDER 1978.nam
[5/21] já concluído, reutilizando: FENDER ELT 2018.nam
[6/21] já concluído, reutilizando: FENDER ULTRA 2.nam
[7/21] já concluído, reutilizando: FODERA ELITE DLX.nam
[8/21] já concluído, reutilizando: FODERA.nam
[9/21] já concluído, reutilizando: G&L L-2500 Americano.nam
[10/21] já concluído, reutilizando: KEN SMITH.nam
[11/21] já concluído, reutilizando: LAKLAND SL 44-75.nam
[12/21] já concluído, reutilizando: MAYONES JABBA.nam
[13/21] já concluído, reutilizando: MTD 535-24.nam
[14/21] já concluído, reutilizando: MTD KING ZX.nam
[15/21] já con